**<h3 style="color:orange">Please make sure any LLM coding assistant like Github Copilot or Cursus is turned OFF!</h2>**

**<h4 style="color:orange">We want you to learn and think critically, not to let LLMs do your work for you.</h4>**

**<h4 style="color:orange">Ask your group members or the internet for help, BEFORE asking your favorite chatbot.</h4>**

<img src="../figures/copilot.png">

# Assignment 1 Part 2: Modelling Boids as a Lagrangian System


In [1]:
# Import all the required libraries here
import json
import pathlib
import pickle
import time

from matplotlib import pyplot as plt
import numpy as np
import torch
# whoever did import torch as T
# please don't
# appreciate it
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, Subset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch
import lightning as L


print(f"Success! Running PyTorch {torch.__version__} and Lightning {L.__version__}")
cuda_available = torch.cuda.is_available()
if cuda_available:
    print(f"CUDA is available! Device count: {torch.cuda.device_count()}")
else:
    print("CUDA is not available.")

/home/ivan/uni/MLS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ivan/uni/MLS/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Success! Running PyTorch 2.14.0+cu132 and Lightning 2.6.6
CUDA is available! Device count: 1


In [2]:
def node_features(state):
    """Appends invariant speed to float states [..., 6], returning [..., 7]."""
    speed = state[..., 3:6].norm(dim=-1, keepdim=True)
    return torch.cat((state[..., :6], speed), dim=-1)


def edge_features(x, edge_index):
    """Returns scalar interaction features [E, 5] from float states [N, >=6].

    Features are distance, relative speed squared, velocity dot product, and
    the source and target velocities projected onto the separation direction.
    """
    src, dst = edge_index
    rel = x[src, :3] - x[dst, :3]  # [E, 3]
    dist = rel.norm(dim=-1, keepdim=True)
    direction = rel / dist.clamp_min(1e-6)
    v_src, v_dst = x[src, 3:6], x[dst, 3:6]
    return torch.cat(
        (
            dist,
            (v_src - v_dst).square().sum(dim=-1, keepdim=True),
            (v_src * v_dst).sum(dim=-1, keepdim=True),
            (v_src * direction).sum(dim=-1, keepdim=True),
            (v_dst * direction).sum(dim=-1, keepdim=True),
        ),
        dim=-1,
    )

In [3]:
class BoidsDataset(Dataset):
    """Boids positions and backward-difference velocities."""

    def __init__(self, path):
        self.path = path
        self.nodes = torch.empty(0)
        self.edge_index = torch.empty((2, 0), dtype=torch.long)
        self.n_simulations = 0
        self.traj_length = 0

    def load_data(self):
        if self.nodes.numel():
            return
        with open(self.path, "rb") as file:
            positions = torch.tensor(np.stack(pickle.load(file)), dtype=torch.float32)

        velocities = positions.diff(dim=1)  # [S, 49, N, 3]
        self.nodes = torch.cat((positions[:, 1:], velocities), dim=-1)  # [S, 49, N, 6]
        self.n_simulations = self.nodes.shape[0]
        self.traj_length = self.nodes.shape[1] - 1
        self.edge_index = (~torch.eye(self.nodes.shape[2], dtype=torch.bool)).nonzero().T


class BoidsTrajectoryDataset(BoidsDataset):
    """One sample contains [N, T, 7], with nodes on the PyG batching axis."""

    def __len__(self):
        return self.n_simulations

    def __getitem__(self, idx):
        x = node_features(self.nodes[idx]).transpose(0, 1)
        return Data(
            x=x,
            edge_index=self.edge_index,
            edge_attr=edge_features(x[:, 0], self.edge_index),
            num_nodes=x.shape[0],
        )


class BoidsStepDataset(BoidsDataset):
    """One sample contains current features [N, 7] and next-state targets [N, 6]."""

    def __len__(self):
        return self.n_simulations * self.traj_length

    def __getitem__(self, idx):
        sim_idx, t_idx = divmod(idx, self.traj_length)
        x = node_features(self.nodes[sim_idx, t_idx])
        return Data(
            x=x,
            y=self.nodes[sim_idx, t_idx + 1],
            edge_index=self.edge_index,
            edge_attr=edge_features(x, self.edge_index),
            num_nodes=x.shape[0],
        )

In [4]:
class DataModule(L.LightningDataModule):
    """Splits whole simulations before exposing their individual timesteps."""

    def __init__(
        self,
        dataset,
        batch_size=16,
        train_split=0.7,
        val_split=0.15,
        seed=42,
    ):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size
        self.train_split = train_split
        self.val_split = val_split
        self.seed = seed

    def setup(self, stage=None):
        """Uses the same seeded simulation split for step and trajectory views."""
        if hasattr(self, "train_set"):
            return
        self.dataset.load_data()
        n = self.dataset.n_simulations
        n_train, n_val = int(n * self.train_split), int(n * self.val_split)
        generator = torch.Generator().manual_seed(self.seed)
        order = torch.randperm(n, generator=generator)
        self.simulation_ids = (
            order[:n_train],
            order[n_train:n_train + n_val],
            order[n_train + n_val:],
        )
        subsets = []
        for ids in self.simulation_ids:
            if isinstance(self.dataset, BoidsStepDataset):
                times = torch.arange(self.dataset.traj_length)
                ids = (ids[:, None] * self.dataset.traj_length + times).flatten()
            subsets.append(Subset(self.dataset, ids.tolist()))
        self.train_set, self.val_set, self.test_set = subsets

    def train_dataloader(self):
        return DataLoader(
            self.train_set, batch_size=self.batch_size, shuffle=True,
            num_workers=2, persistent_workers=True,
            generator=torch.Generator().manual_seed(self.seed),
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, batch_size=self.batch_size,
            num_workers=2, persistent_workers=True,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_set, batch_size=self.batch_size,
            num_workers=2, persistent_workers=True,
        )

    def predict_dataloader(self):
        return self.test_dataloader()

In [5]:
class EGNNBlock(nn.Module):
    """Residual message-passing block on invariant node features.

    Args:
        d_model: hidden feature dimension per node
        edge_dim: number of scalar edge features
    """

    def __init__(self, d_model, edge_dim=5):
        super().__init__()

        self._d_model = d_model

        # Messages
        self.norm = nn.LayerNorm(d_model)
        self.message_mlp = nn.Sequential(
            nn.Linear(2 * d_model + edge_dim, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
        )

        # Node update
        self.node_mlp = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, h, edge_index, edge_attr) -> tuple[torch.Tensor, torch.Tensor]:
        """Maps h [N, D] and edges [E, 5] to updated h and messages [E, D]."""
        # Messages
        src, dst = edge_index
        z = self.norm(h)  # [N, D]
        m = self.message_mlp(
            torch.cat((z[src], z[dst], edge_attr), dim=-1)
        )  # [E, 2D + 5] -> [E, D]

        # Aggregate
        agg = h.new_zeros(h.shape).index_add_(0, dst, m)  # [N, D]
        degree = torch.bincount(dst, minlength=h.shape[0]).unsqueeze(-1)  # [N, 1]
        agg = agg / degree.clamp_min(1)

        # Node update
        h = h + self.node_mlp(torch.cat((z, agg), dim=-1))  # [N, D]
        return h, m

In [6]:
class EGNN(nn.Module):
    """Predicts next velocity using E(3)-equivariant vector updates.

    Scalar MLPs weight relative positions, relative velocities and the current
    velocity. These vectors transform under rotations and reflections, while
    the scalar coefficients remain unchanged. Positions enter through pairwise
    differences, preserving translation symmetry.

    Args:
        d_model: hidden feature dimension per node
        n_layers: number of message-passing blocks
    """

    def __init__(self, d_model=64, n_layers=3):
        super().__init__()
        self._d_model = d_model
        self._n_layers = n_layers

        # Embedding
        self.input_proj = nn.Linear(1, d_model)

        # Blocks
        self.blocks = nn.ModuleList([
            EGNNBlock(d_model) for _ in range(n_layers)
        ])

        # Vector coefficients
        self.edge_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, 2),
        )
        self.velocity_proj = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1),
        )
        # start near constant velocity while retaining gradients through every block
        for head in (self.edge_proj[-1], self.velocity_proj[-1]):
            nn.init.xavier_uniform_(head.weight, gain=0.001)
            nn.init.zeros_(head.bias)

    def forward(self, x, edge_index, edge_attr) -> torch.Tensor:
        """Maps float states [N, 7] and long edges [2, E] to velocities [N, 3]."""
        # Embedding
        v = x[:, 3:6]  # [N, 3]
        h = self.input_proj(x[:, 6:7])  # [N, 1] -> [N, D]

        # Blocks
        for block in self.blocks:
            h, m = block(h, edge_index, edge_attr)

        # Vector update
        src, dst = edge_index
        rel = x[src, :3] - x[dst, :3]
        # bounded separation vectors avoid amplifying very distant pairs
        rel = rel / (1 + rel.norm(dim=-1, keepdim=True))
        weights = self.edge_proj(m)  # [E, 2]
        dv = weights[:, :1] * rel + weights[:, 1:] * (v[src] - v[dst])
        agg = v.new_zeros(v.shape).index_add_(0, dst, dv)
        degree = torch.bincount(dst, minlength=x.shape[0]).clamp_min(1).unsqueeze(-1)
        return v + self.velocity_proj(h) * v + agg / degree  # [N, 3]

In [7]:
class BoidsModel(L.LightningModule):
    """Trains on adjacent states and rolls out from one observed initial state."""

    def __init__(self, model, learning_rate=5e-4):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, x, edge_index, edge_attr) -> torch.Tensor:
        """Returns next-state features [N, 7] from current features [N, 7]."""
        v = self.model(x, edge_index, edge_attr)
        return node_features(torch.cat((x[:, :3] + v, v), dim=-1))

    def _step(self, batch):
        pred = self(batch.x, batch.edge_index, batch.edge_attr)
        return F.mse_loss(pred[:, 3:6], batch.y[:, 3:6])

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=batch.num_graphs)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=batch.num_graphs)
        return loss

    def test_step(self, batch, batch_idx):
        pred = self(batch.x, batch.edge_index, batch.edge_attr)
        loss = F.mse_loss(pred[:, 3:6], batch.y[:, 3:6])
        self.log("test_loss", loss, batch_size=batch.num_graphs)
        for name, channels in (("position", slice(0, 3)), ("velocity", slice(3, 6))):
            mse = F.mse_loss(pred[:, channels], batch.y[:, channels])
            self.log(f"test_{name}_mse", mse, batch_size=batch.num_graphs)
        return loss

    def predict_step(self, batch, batch_idx):
        """Returns prediction and target [B, T, N, 6], paired in the same batch."""
        x = batch.x[:, 0]
        states = [x[:, :6]]
        for _ in range(batch.x.shape[1] - 1):
            # topology is complete and fixed; geometric features change every step
            x = self(x, batch.edge_index, edge_features(x, batch.edge_index))
            states.append(x[:, :6])
        pred, _ = to_dense_batch(torch.stack(states, dim=1), batch.batch)
        target, _ = to_dense_batch(batch.x[..., :6], batch.batch)
        return {
            "pred": pred.transpose(1, 2).cpu(),  # [B, T, N, 6]
            "target": target.transpose(1, 2).cpu(),
        }

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

In [8]:
class LossHistory(L.Callback):
    """Stores epoch losses, excluding Lightning's validation sanity check."""

    def __init__(self):
        self.train_loss = []
        self.val_loss = []

    def on_train_epoch_end(self, trainer, pl_module):
        self.train_loss.append(float(trainer.callback_metrics["train_loss"]))
        print(
            f"Epoch {trainer.current_epoch + 1}: "
            f"train={self.train_loss[-1]:.6f}, val={self.val_loss[-1]:.6f}",
            flush=True,
        )
        with open(OUTPUT_DIR / "losses.json", "w") as file:
            json.dump({"train": self.train_loss, "val": self.val_loss}, file)

    def on_validation_epoch_end(self, trainer, pl_module):
        if not trainer.sanity_checking:
            self.val_loss.append(float(trainer.callback_metrics["val_loss"]))

In [9]:
def kinematics(states):
    """Computes flock properties [S, T] from CPU float states [S, T, N, 6]."""
    pos, vel = states[..., :3], states[..., 3:6]
    speed = vel.norm(dim=-1)
    heading = F.normalize(vel, dim=-1, eps=1e-6)
    centered = pos - pos.mean(dim=2, keepdim=True)
    # loop over simulations to avoid allocating [S, T, N, N] at once
    nearest = []
    for trajectory in pos:
        distances = torch.cdist(trajectory, trajectory)
        distances.diagonal(dim1=-2, dim2=-1).fill_(float("inf"))
        nearest.append(distances.min(dim=-1).values.mean(dim=-1))
    return {
        "Speed": speed.mean(dim=-1),
        "Acceleration": vel.diff(dim=1).norm(dim=-1).mean(dim=-1),
        "Polarization": heading.mean(dim=2).norm(dim=-1),
        "Flock radius": centered.square().sum(dim=-1).mean(dim=-1).sqrt(),
        "Nearest-neighbor distance": torch.stack(nearest),
        "Distance travelled": pos.diff(dim=1).norm(dim=-1).cumsum(dim=1).mean(dim=-1),
    }

In [10]:
def plot_properties(series, filename):
    """Plots temporal means and standard deviations across simulations."""
    fig, axes = plt.subplots(2, 3, figsize=(13, 7), layout="constrained")
    for ax, name in zip(axes.flat, next(iter(series.values()))):
        for label, properties in series.items():
            values = properties[name]
            start = 1 if name in ("Acceleration", "Distance travelled") else 0
            t = np.arange(start, start + values.shape[1])
            mean = values.mean(dim=0).numpy()
            std = values.std(dim=0, unbiased=False).numpy()
            line, = ax.plot(t, mean, label=label)
            ax.fill_between(t, mean - std, mean + std,
                            color=line.get_color(), alpha=0.12)
        ax.set(xlabel="Saved frame", ylabel=name)
        ax.legend()
    fig.savefig(OUTPUT_DIR / filename, dpi=160)
    plt.close(fig)

In [11]:
def evaluate_rollouts(pred, target):
    """Measures forecast error and explicitly records divergent trajectories."""
    horizon = torch.arange(target.shape[1], dtype=target.dtype)[None, :, None, None]
    baseline = target[:, :1].expand_as(target).clone()
    baseline[..., :3] += horizon * target[:, :1, :, 3:6]
    finite = torch.isfinite(pred).all(dim=(2, 3))  # [S, T]
    survivors = finite.all(dim=1)
    metrics = {
        "finite_trajectories": finite.sum(dim=0).tolist(),
        "surviving_trajectories": survivors.sum().item(),
        "test_trajectories": pred.shape[0],
    }

    # Errors over the complete test cohort; failed forecasts keep undefined scores
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for label, states in (("EGNN", pred), ("Constant velocity", baseline)):
        errors = (states[:, 1:, :, :3].double() - target[:, 1:, :, :3]).norm(dim=-1)
        curve = errors.mean(dim=(0, 2))
        metrics[label] = {
            "ADE": errors.mean().item() if torch.isfinite(errors).all() else None,
            "FDE": errors[:, -1].mean().item() if torch.isfinite(errors[:, -1]).all() else None,
            "ADE_first_10": errors[:, :10].mean().item(),
            "DE_step_10": curve[9].item(),
            "displacement_curve": [v if np.isfinite(v) else None for v in curve.tolist()],
        }
        axes[0].plot(np.arange(1, 11), curve[:10], label=label)
        axes[1].plot(np.arange(1, target.shape[1]), curve, label=label)
        print(f"{label}: {metrics[label]}")
    axes[0].set(title="First 10 steps, all 300 trajectories")
    axes[1].set(title="Full-cohort mean until numerical failure", yscale="log")
    for ax in axes:
        ax.set(xlabel="Rollout step", ylabel="Displacement error")
        ax.legend()
    fig.savefig(OUTPUT_DIR / "displacement_error.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
    ax.plot(finite.sum(dim=0))
    ax.set(xlabel="Rollout step", ylabel="Finite trajectories",
           ylim=(0, pred.shape[0] + 5))
    fig.savefig(OUTPUT_DIR / "rollout_failures.png", dpi=160)
    plt.close(fig)

    # Short-horizon behavior uses all trajectories, before any numerical failures
    properties = {
        "EGNN": kinematics(pred[:, :11]),
        "Target": kinematics(target[:, :11]),
        "Constant velocity": kinematics(baseline[:, :11]),
    }
    plot_properties(properties, "rollout_properties.png")
    metrics["properties_first_10"] = summarize_properties(properties)
    for label, states in (("EGNN", pred), ("Target", target), ("Constant velocity", baseline)):
        outside = ((states[:, 1:11, :, :3] < 0) | (states[:, 1:11, :, :3] > 25)).any(dim=-1)
        metrics.setdefault(label, {})["outside_box_first_10"] = outside.float().mean().item()

    # Conditional diagnostics explain what happens to the finite survivors
    properties = {
        "EGNN (finite survivors)": kinematics(pred[survivors]),
        "Target (same simulations)": kinematics(target[survivors]),
    }
    plot_properties(properties, "survivor_properties.png")
    errors = (pred[survivors, 1:, :, :3] - target[survivors, 1:, :, :3]).norm(dim=-1)
    metrics["survivors"] = {
        "ADE": errors.mean().item(),
        "FDE": errors[:, -1].mean().item(),
        "properties": summarize_properties(properties),
    }

    # The first test example, restricted to the first 10 forecast steps
    fig = plt.figure(figsize=(12, 4), layout="constrained")
    all_pos = torch.cat((target[0, :11, :, :3], pred[0, :11, :, :3]), dim=0)
    limits = [(float(all_pos[..., i].min()), float(all_pos[..., i].max()))
              for i in range(3)]
    for panel, (label, states) in enumerate((("Target", target), ("EGNN", pred)), 1):
        ax = fig.add_subplot(1, 2, panel, projection="3d")
        for particle in range(12):
            ax.plot(*states[0, :11, particle, :3].numpy().T, alpha=0.7)
        ax.set(title=f"{label}: first 10 steps", xlabel="x", ylabel="y", zlabel="z",
               xlim=limits[0], ylim=limits[1], zlim=limits[2])
    fig.savefig(OUTPUT_DIR / "trajectories.png", dpi=160)
    plt.close(fig)
    return metrics

In [12]:
def summarize_properties(series):
    """Returns temporal means and population spreads across simulations."""
    return {
        label: {
            name: {
                "mean": values.mean(dim=0).tolist(),
                "std": values.std(dim=0, unbiased=False).tolist(),
            }
            for name, values in properties.items()
        }
        for label, properties in series.items()
    }

def benchmark(model, batch):
    """Times a warm one-step GPU forward pass, excluding loading and transfers."""
    model.eval()
    with torch.inference_mode():
        for _ in range(10):
            model(batch.x, batch.edge_index, batch.edge_attr)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(100):
            model(batch.x, batch.edge_index, batch.edge_attr)
        torch.cuda.synchronize()
        milliseconds = 1000 * (time.perf_counter() - start) / 100
        with torch.profiler.profile(activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ]) as profile:
            for _ in range(20):
                model(batch.x, batch.edge_index, batch.edge_attr)
            torch.cuda.synchronize()
    with open(OUTPUT_DIR / "profile.txt", "w") as file:
        file.write(profile.key_averages().table(sort_by="self_cuda_time_total", row_limit=20))
    return {"forward_ms": milliseconds, "batch_size": batch.num_graphs}

In [13]:
PATH = pathlib.Path.cwd().parent / "data" / "boids_dataset.pkl"
BATCH_SIZE = 16
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
MAX_EPOCHS = 40
LIMIT_TRAIN_BATCHES = 300
OUTPUT_DIR = pathlib.Path.cwd() / "boids_results"
LEARNING_RATE = 5e-4

In [14]:
L.seed_everything(42, workers=True)
torch.set_num_threads(1)
OUTPUT_DIR.mkdir(exist_ok=True)

# Data
dataset = BoidsStepDataset(PATH)
datamodule = DataModule(dataset, BATCH_SIZE, TRAIN_SPLIT, VAL_SPLIT)
datamodule.setup()
train_ids = datamodule.simulation_ids[0]
eda = {"Train": kinematics(dataset.nodes[train_ids])}
plot_properties(eda, "eda.png")
with open(OUTPUT_DIR / "eda.json", "w") as file:
    json.dump(summarize_properties(eda), file, indent=2)

# Model
egnn = EGNN(d_model=64, n_layers=3)
model = BoidsModel(egnn, learning_rate=LEARNING_RATE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
loss_history = LossHistory()
trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    limit_train_batches=LIMIT_TRAIN_BATCHES,
    enable_progress_bar=False,
    accelerator="auto",
    devices=1,
    callbacks=[loss_history],
    logger=False,
    enable_checkpointing=False,
    gradient_clip_val=1.0,
)

# Train
start = time.perf_counter()
trainer.fit(model, datamodule=datamodule)
training_seconds = time.perf_counter() - start
print(f"Training time: {training_seconds:.1f} s")
torch.save(model.state_dict(), OUTPUT_DIR / "model.pt")
one_step = trainer.test(model, datamodule=datamodule)

fig, ax = plt.subplots(figsize=(6, 4))
epochs = np.arange(1, MAX_EPOCHS + 1)
ax.plot(epochs, loss_history.train_loss, label="Train")
ax.plot(epochs, loss_history.val_loss, label="Validation")
ax.set(xlabel="Epoch", ylabel="Velocity MSE", yscale="log")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "loss_curves.png", dpi=160)
plt.close(fig)

# Rollout
traj_dataset = BoidsTrajectoryDataset(PATH)
traj_module = DataModule(traj_dataset, BATCH_SIZE, TRAIN_SPLIT, VAL_SPLIT)
start = time.perf_counter()
rollouts = trainer.predict(model, datamodule=traj_module)
rollout_seconds = time.perf_counter() - start
print(f"Rollout time: {rollout_seconds:.1f} s")
pred = torch.cat([batch["pred"] for batch in rollouts])
target = torch.cat([batch["target"] for batch in rollouts])
torch.save({"pred": pred, "target": target}, OUTPUT_DIR / "rollouts.pt")
metrics = evaluate_rollouts(pred, target)
batch = next(iter(datamodule.test_dataloader())).to("cuda")
model = model.to("cuda")
timing = benchmark(model, batch)
with open(OUTPUT_DIR / "metrics.json", "w") as file:
    json.dump({
        "one_step": one_step,
        "rollout": metrics,
        "training_seconds": training_seconds,
        "rollout_seconds": rollout_seconds,
        "timing": timing,
        "epochs": MAX_EPOCHS,
        "train_batches_per_epoch": LIMIT_TRAIN_BATCHES,
        "batch_size": BATCH_SIZE,
        "seed": 42,
        "parameters": sum(p.numel() for p in model.parameters()),
        "test_simulations": traj_module.simulation_ids[2].tolist(),
    }, file, indent=2)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Parameters: 80,451


┏━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ EGNN │ 80.5 K │ train │     0 │
└───┴───────┴──────┴────────┴───────┴───────┘

Trainable params: 80.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 80.5 K                                                                                               
Total estimated model params size (MB): 0.322                                                                      
Modules in train mode: 43                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/ivan/uni/MLS/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Training time: 463.8 s


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   0.010522772558033466    │
│     test_position_mse     │   0.010522773489356041    │
│     test_velocity_mse     │   0.010522772558033466    │
└───────────────────────────┴───────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Rollout time: 4.9 s
EGNN: {'ADE': None, 'FDE': None, 'ADE_first_10': 3.820606673960337, 'DE_step_10': 7.776455972717435, 'displacement_curve': [0.2619250263549017, 0.711290877244601, 1.441322131552611, 2.3257090747447, 3.2723698606135074, 4.233600249514928, 5.168914029331155, 6.07231816135592, 6.942161356173593, 7.776455972717435, 8.570206687591407, 9.328950159544668, 32.179579642006985, 27046564981.459248, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]}
Constant velocity: {'ADE': 34.3017300218482, 'FDE': 67.18092653985151, 'ADE_first_10': 6.536922535511681, 'DE_step_10': 13.310736815702262, 'displacement_curve': [0.6292526636059107, 1.592163495431902, 2.771914602027618, 4.0985178954005015, 5.526955431690851, 7.023573328906622, 8.565188643263172, 10.133675332376367, 11.717247146711609, 13.310736815702262, 14.909294900348005, 16.507

USDT:2026-09-25 21:17:13 298822:298822 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-25 21:17:14 298822:298822 SyncActivityProfilerHandler.cpp:46] profiler_stop
